# 01 · Ingest Data BTS → Parquet

**Langkah [1]–[2] dari alur aplikasi** (lihat `PRD.md` §4).

Notebook ini:
1. Mengunduh data **BTS Airline On-Time Performance** per bulan (2024–2025, 24 file ZIP ± 28 MB).
2. Mengambil kolom yang dibutuhkan saja (dari 110 kolom).
3. Menyimpan setiap bulan sebagai **Parquet** di Google Drive: `parquet/year=YYYY/month=MM.parquet`.
4. Membuat ringkasan `ingest_stats.json` (jumlah baris, ukuran) yang nanti dipakai animasi *Cara Kerja* di web.

Aman dijalankan ulang: bulan yang sudah jadi Parquet akan dilewati. Kalau Colab terputus, jalankan ulang saja.

## 1. Hubungkan Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Data BTS 2024-2025'  # folder bersama tim
    ZIP_DIR = '/content/zips'                           # disk sementara Colab, bukan Drive
except ImportError:  # dijalankan di laptop
    BASE = './data'
    ZIP_DIR = './data/zips'

## 2. Konfigurasi

In [ ]:
import os, json, time, zipfile, urllib.request
import pandas as pd

YEARS = [2024, 2025]
URL = 'https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{y}_{m}.zip'
PARQUET_DIR = os.path.join(BASE, 'parquet')

# Kolom yang diambil + tipe data hemat memori.
# *Delay penyebab (CarrierDelay, dst.) hanya untuk EDA — JANGAN dipakai sebagai fitur model (bocoran/leakage).
COLUMNS = {
    'FlightDate': 'string',
    'Year': 'int16', 'Month': 'int8', 'DayofMonth': 'int8', 'DayOfWeek': 'int8',
    'Reporting_Airline': 'category',
    'Flight_Number_Reporting_Airline': 'int32',
    'Tail_Number': 'string',
    'Origin': 'category', 'OriginState': 'category',
    'Dest': 'category', 'DestState': 'category',
    'CRSDepTime': 'int16',   # jadwal berangkat HHMM, mis. 856 = 08:56
    'CRSArrTime': 'int16',
    'CRSElapsedTime': 'float32',
    'Distance': 'float32',
    'DepDelay': 'float32', 'DepDelayMinutes': 'float32',
    'ArrDelay': 'float32', 'ArrDelayMinutes': 'float32',
    'Cancelled': 'float32', 'CancellationCode': 'category', 'Diverted': 'float32',
    'CarrierDelay': 'float32', 'WeatherDelay': 'float32', 'NASDelay': 'float32',
    'SecurityDelay': 'float32', 'LateAircraftDelay': 'float32',
}

os.makedirs(PARQUET_DIR, exist_ok=True)
os.makedirs(ZIP_DIR, exist_ok=True)

## 3. Fungsi unduh & konversi

In [ ]:
def parquet_path(y, m):
    return os.path.join(PARQUET_DIR, f'year={y}', f'month={m:02d}.parquet')

def download_zip(y, m, retries=3):
    path = os.path.join(ZIP_DIR, f'{y}_{m}.zip')
    if os.path.exists(path):
        return path
    for i in range(retries):
        try:
            tmp = path + '.part'
            urllib.request.urlretrieve(URL.format(y=y, m=m), tmp)
            os.replace(tmp, path)
            return path
        except Exception as e:
            print(f'  gagal unduh ({e}), coba lagi...')
            time.sleep(5 * (i + 1))
    raise RuntimeError(f'Tidak bisa mengunduh {y}-{m:02d}')

def convert_month(y, m, keep_zip=False):
    out = parquet_path(y, m)
    if os.path.exists(out):
        return 'skip'
    zp = download_zip(y, m)
    # CSV dibaca langsung dari ZIP (isinya CSV + readme.html); hanya kolom terpilih yang dimuat ke memori
    with zipfile.ZipFile(zp) as z:
        name = next(n for n in z.namelist() if n.endswith('.csv'))
        with z.open(name) as f:
            df = pd.read_csv(f, usecols=list(COLUMNS), dtype=COLUMNS, encoding='latin-1')
    assert (df['Year'] == y).all() and (df['Month'] == m).all(), 'isi file tidak sesuai bulan'
    os.makedirs(os.path.dirname(out), exist_ok=True)
    df.to_parquet(out + '.part', index=False, compression='zstd')
    os.replace(out + '.part', out)  # tulis atomik: file setengah jadi tidak dianggap selesai
    if not keep_zip:
        os.remove(zp)
    return len(df)

## 4. Jalankan untuk semua bulan (± 1–2 menit per bulan)

In [ ]:
for y in YEARS:
    for m in range(1, 13):
        t = time.time()
        r = convert_month(y, m)
        print(f'{y}-{m:02d}:', 'sudah ada, dilewati' if r == 'skip' else f'{r:,} baris ({time.time() - t:.0f} detik)')

## 5. Validasi & ringkasan (DuckDB membaca semua Parquet sekaligus)

In [ ]:
import duckdb

glob = os.path.join(PARQUET_DIR, '*', '*.parquet').replace('\\', '/')
con = duckdb.connect()
per_month = con.sql(f'''
    SELECT Year, Month, COUNT(*) AS flights,
           SUM(Cancelled)::INT AS cancelled,
           SUM(Diverted)::INT AS diverted,
           ROUND(AVG(DepDelayMinutes), 2) AS avg_dep_delay
    FROM read_parquet('{glob}')
    GROUP BY 1, 2 ORDER BY 1, 2
''').df()
display(per_month) if 'display' in globals() else print(per_month)

missing = [(y, m) for y in YEARS for m in range(1, 13) if not os.path.exists(parquet_path(y, m))]
assert not missing, f'Bulan belum ada: {missing}'

In [ ]:
files = [parquet_path(y, m) for y in YEARS for m in range(1, 13)]
stats = {
    'period': f'{YEARS[0]}-{YEARS[-1]}',
    'months': len(files),
    'raw_columns': 110,
    'kept_columns': len(COLUMNS),
    'rows': int(per_month.flights.sum()),
    'cancelled': int(per_month.cancelled.sum()),
    'diverted': int(per_month.diverted.sum()),
    'parquet_mb': round(sum(os.path.getsize(f) for f in files) / 1e6, 1),
    'generated_at': time.strftime('%Y-%m-%d %H:%M'),
}
with open(os.path.join(BASE, 'ingest_stats.json'), 'w') as f:
    json.dump(stats, f, indent=2)
stats

## 6. Intip data

In [ ]:
con.sql(f"SELECT * FROM read_parquet('{glob}') LIMIT 5").df()